In [ ]:
import os
import zarr
import s3fs
import dask
import xarray as xr
from rechunker import rechunk
from dask import compute
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
from dask.distributed import Client, progress

# For connecting to dashboard

In [ ]:
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Allocate the resources using slurm cluster

In [ ]:
cluster = SLURMCluster(name='dask-cluster',
                      cores=1,
                      memory='8GB', # 👈 Can be tuned
                      processes=1,
                      interface='ib0',
                      queue='compute',
                      account='bk1414',
                      walltime='08:00:00', # 👈 Max. time limit on 'compute'
                      asynchronous=0)
cluster.scale(jobs=15)  # 👈 number of SLURM jobs .. max. limit on levante is 20 per user, but 1 job is consumed for 'jupyterlab spawner', so max. 19 
client = Client(cluster) 

In [ ]:
client

# Initialization and store paths etc. 

In [ ]:
workingDir            = '/work/bk1414/k204247/S3toHSM/nextgems' # 👈 Change path accordingly
prodEndpointURL       = 'https://s3.eu-dkrz-3.dkrz.cloud' # 👈 Change path accordingly
srcPath               = 'nextgems/' # 👈 Change path accordingly
currentSrcStore       = 'nextgems/ngc4008_PT15M_9.zarr' # 👈 Change path accordingly

zoomLevelCells        = [  ]
for z in range (0,11):
    if z == 0:
        zoomLevelCells.extend( [12] )
    else:
        prev = zoomLevelCells[ z - 1 ]
        zoomLevelCells.extend(  [prev * 4] )

In [ ]:
srcS3Cluster = s3fs.S3FileSystem(
    anon=True,  # 👈 if credentials needed,set False and provide them.
    endpoint_url=prodEndpointURL,
    config_kwargs={"max_pool_connections": 150},  # parallel reads  # 👈 tune this as per the S3 setup
)

store = s3fs.S3Map( currentSrcStore, s3 = srcS3Cluster, check = False )
source_group = zarr.open_consolidated( store, mode='r', zarr_version = 2 ) 
print(source_group.tree())

# Rechunker settings

In [ ]:
src_storeName = currentSrcStore.split(os.path.sep)[-1]
src_zoomLevel = int( src_storeName.split('.')[0].split('_')[-1] )
target_chunks_dict = {
                            "tas" : ( 1, zoomLevelCells[ src_zoomLevel ]  ),   #{ "time" : 1 , "cell" : 12 }
                            "pr"  : ( 1, zoomLevelCells[ src_zoomLevel ]  ),   #{ "time" : 1 , "cell" : 12 }

}

target_store  = f"nextgemsSingleTimestep/{ src_storeName }"
temp_store    = f"nextgemsSingleTimestep/{ src_storeName.split('.')[0] }_temp.zarr"
target_root = zarr.open_group(target_store, mode="w")
temp_root = zarr.open_group(temp_store, mode="w")

In [ ]:
plans = []

for name in source_group.array_keys():
    array = source_group[name]

    plan = rechunk(
        array,
        target_chunks=(1, zoomLevelCells[src_zoomLevel]),
        max_mem="2GB", # 👈 Can be tuned
        target_store=f"{target_store}/{name}",
        temp_store=f"{temp_store}/{name}",
    )

    plans.append(plan)

In [ ]:
with dask.config.set(scheduler="distributed"):
    compute(*[plan.execute() for plan in plans], traverse=False)

In [ ]:
zarr.consolidate_metadata( target_store )

In [ ]:
client.close()

In [ ]:
client.status

In [ ]:
# # Verify the rechunked store
target_group = zarr.open( target_store, mode='r' )
print( f"{ [ target_group[arr[0]].info for arr in target_group.arrays() ] }\n" )
print( f"{ xr.open_dataset(target_store) }\n" )